# 🏠 PROPBOT — William Properti RAG Chatbot

Chatbot berbasis RAG (Retrieval-Augmented Generation) untuk agen properti William.
Customer bisa tanya langsung ke AI tentang properti di Jawa Barat.

**Arsitektur:**
```
PDF Katalog Properti
       ↓
  PyPDFLoader → Chunking (NLTKTextSplitter)
       ↓
  GoogleGenerativeAI Embedding → ChromaDB
       ↓
  User tanya → Retriever → Prompt → Gemini → Jawaban
       ↓
  Streamlit UI (via ngrok tunnel)
```

**Sebelum mulai, tambahkan 2 secrets di Colab** (klik ikon 🔑 di sidebar kiri):
- `GEMINI` → API key dari https://aistudio.google.com
- `NGROK_TOKEN` → token dari https://ngrok.com (gratis)

## Cell 1 — Install Library

In [1]:
!pip install -q -U \
    google-generativeai \
    langchain \
    langchain-google-genai \
    langchain_community \
    pypdf \
    chromadb \
    streamlit \
    pyngrok \
    nltk

print('✅ Semua library berhasil diinstall!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8

## Cell 2 — Setup API Key & NLTK

In [2]:
import os
import nltk
from google.colab import userdata

GEMINI_KEY = userdata.get('GEMINI')
NGROK_TOKEN = userdata.get('NGROK_TOKEN')

os.environ['GOOGLE_API_KEY'] = GEMINI_KEY

nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)

print('✅ API Key & NLTK siap!')

✅ API Key & NLTK siap!


## Cell 3 — Upload PDF Katalog Properti

Upload file `Katalog_Properti_Jabar_William.pdf` (atau PDF katalog lainnya).

In [3]:
# Clone repo dari GitHub dan set pdf_filename
!git clone https://github.com/cangkirofficial/propbot-william-properti.git
%cd propbot-william-properti

pdf_filename = 'Katalog_Properti_Jabar_William.pdf'
print(f'✅ Menggunakan file: {pdf_filename}')

with open('pdf_filename.txt', 'w') as f:
    f.write(pdf_filename)

📂 Upload file PDF katalog properti kamu...


Saving Katalog_Properti_Jabar_William.pdf to Katalog_Properti_Jabar_William.pdf
✅ File "Katalog_Properti_Jabar_William.pdf" berhasil diupload!


## Cell 4 — Build RAG System

Proses: Load PDF → Chunking → Embedding → Simpan ke ChromaDB → Buat RAG chain.

In [8]:
import warnings
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import NLTKTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print('📄 Loading PDF...')
loader = PyPDFLoader(pdf_filename)
pages = loader.load_and_split()
print(f'   → {len(pages)} halaman ditemukan')

print('✂️  Chunking dokumen...')
text_splitter = NLTKTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(pages)
print(f'   → {len(chunks)} chunks dibuat')

print('🔢 Membuat embeddings (ini mungkin butuh 1-2 menit)...')
embeddings = GoogleGenerativeAIEmbeddings(
    model='models/gemini-embedding-2',
    google_api_key=GEMINI_KEY,
    output_dimensionality=768
)

print('💾 Menyimpan ke ChromaDB...')
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='./chroma_properti_db'
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 5})
print('   → Vector store berhasil dibuat!')

chat_model = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash-lite',
    google_api_key=GEMINI_KEY,
    temperature=0.3
)

SYSTEM_PROMPT = """Kamu adalah asisten virtual William, seorang agen properti profesional di Jawa Barat.
Nama kamu adalah PROPBOT — asisten cerdas yang membantu calon pembeli menemukan properti impian mereka.

Tugasmu:
1. Jawab pertanyaan tentang properti berdasarkan katalog yang tersedia
2. Bantu bandingkan properti (harga, lokasi, cicilan)
3. Berikan rekomendasi berdasarkan kebutuhan customer
4. Sampaikan informasi kontak William jika customer tertarik lanjut

Gaya bicara:
- Ramah, profesional, dan antusias
- Gunakan Bahasa Indonesia yang baik dan hangat
- Sertakan detail harga dan cicilan saat relevan
- Gunakan emoji secukupnya (🏠 🏡 💰 📍 ✨)
- Jika informasi tidak ada di katalog, katakan jujur dan arahkan ke William

Kontak William: 📞 123456789

PENTING: Hanya gunakan informasi dari konteks. Jangan mengarang data properti."""

chat_template = ChatPromptTemplate.from_messages([
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessagePromptTemplate.from_template("""
Konteks katalog properti:
{context}

Pertanyaan: {question}

Jawaban:""")
])

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | chat_template
    | chat_model
    | StrOutputParser()
)

print('\n🧪 Test RAG chain...')
test = rag_chain.invoke('Properti apa saja yang tersedia di Bandung?')
print('Contoh jawaban:', test[:300], '...')
print('\n✅ RAG system siap!')

📄 Loading PDF...
   → 4 halaman ditemukan
✂️  Chunking dokumen...
   → 4 chunks dibuat
🔢 Membuat embeddings (ini mungkin butuh 1-2 menit)...
💾 Menyimpan ke ChromaDB...
   → Vector store berhasil dibuat!

🧪 Test RAG chain...
Contoh jawaban: Halo! Saya PROPBOT, asisten virtual William. Dengan senang hati saya akan bantu Anda menemukan properti impian di Bandung! ✨

Berikut adalah properti yang tersedia di Bandung:

*   **Cluster Emerald**
    *   Harga: Rp 1.2M
    *   Landmark: Dekat Trans Studio Mall
    *   Simulasi Cicilan: DP 10%,  ...

✅ RAG system siap!


## Cell 5 — Tulis File Streamlit App

Cell ini menulis file `propbot_app.py` ke disk — ini adalah UI chatbot yang akan dijalankan Streamlit.

In [11]:
%%writefile propbot_app.py
import streamlit as st
import os
import warnings
warnings.filterwarnings("ignore")

st.set_page_config(
    page_title="William Properti — PROPBOT",
    page_icon="🏠",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
    @import url("https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;700&family=Lato:wght@300;400;700&display=swap");
    .stApp { background: linear-gradient(135deg, #0a0a0a 0%, #1a1410 50%, #0f0d0a 100%); font-family: "Lato", sans-serif; }
    .main .block-container { max-width: 900px; padding-top: 2rem; }
    .header-banner { background: linear-gradient(135deg, #1a1208 0%, #2d1f0a 50%, #1a1208 100%); border: 1px solid #c9a84c; border-radius: 16px; padding: 2rem 2.5rem; margin-bottom: 2rem; }
    .header-title { font-family: "Playfair Display", serif; font-size: 2.2rem; font-weight: 700; color: #c9a84c; margin: 0; }
    .header-subtitle { font-size: 0.9rem; color: #9b8c6e; margin-top: 0.3rem; letter-spacing: 0.1em; text-transform: uppercase; }
    .header-badge { display: inline-block; background: rgba(201,168,76,0.15); border: 1px solid rgba(201,168,76,0.4); color: #c9a84c; font-size: 0.7rem; padding: 0.2rem 0.7rem; border-radius: 20px; letter-spacing: 0.1em; text-transform: uppercase; margin-top: 0.8rem; }
    [data-testid="stSidebar"] { background: linear-gradient(180deg, #0d0b08 0%, #1a1208 100%) !important; border-right: 1px solid rgba(201,168,76,0.2) !important; }
    [data-testid="stSidebar"] * { color: #c9a84c !important; }
    .contact-card { background: rgba(201,168,76,0.08); border: 1px solid rgba(201,168,76,0.25); border-radius: 12px; padding: 1rem; margin: 0.5rem 0 1rem; }
    [data-testid="stChatMessage"][data-role="user"] { background: rgba(201,168,76,0.08) !important; border: 1px solid rgba(201,168,76,0.2) !important; border-radius: 16px 4px 16px 16px !important; }
    [data-testid="stChatMessage"][data-role="assistant"] { background: rgba(255,255,255,0.03) !important; border: 1px solid rgba(255,255,255,0.07) !important; border-radius: 4px 16px 16px 16px !important; }
    .stChatInputContainer { border: 1px solid rgba(201,168,76,0.3) !important; border-radius: 12px !important; }
    .stChatInputContainer:focus-within { border-color: #c9a84c !important; }
    .stMarkdown p, .stMarkdown li { color: #c8b99a !important; }
    #MainMenu {visibility: hidden;} footer {visibility: hidden;} header {visibility: hidden;}
</style>
""", unsafe_allow_html=True)

@st.cache_resource(show_spinner=False)
def load_rag():
    import nltk
    nltk.download("punkt_tab", quiet=True)
    nltk.download("punkt", quiet=True)
    from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
    from langchain_community.vectorstores import Chroma
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.messages import SystemMessage
    from langchain_core.prompts import HumanMessagePromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.runnables import RunnablePassthrough

    api_key = os.environ.get("GOOGLE_API_KEY", "")

    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-2",
        google_api_key=api_key,
        output_dimensionality=768
    )
    vectorstore = Chroma(
        persist_directory="./chroma_properti_db",
        embedding_function=embeddings
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    chat_model = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash-lite",
        google_api_key=api_key,
        temperature=0.3
    )

    SYSTEM_PROMPT = (
        "Kamu adalah asisten virtual William, agen properti profesional di Jawa Barat. "
        "Nama kamu adalah PROPBOT. Bantu calon pembeli temukan properti impian mereka. "
        "Jawab ramah, profesional, Bahasa Indonesia yang hangat. "
        "Sertakan harga dan cicilan saat relevan. Gunakan emoji secukupnya. "
        "Kontak William: 123456789. "
        "PENTING: Hanya gunakan info dari konteks. Jangan mengarang data properti."
    )

    HUMAN_TEMPLATE = (
        "Konteks katalog:\n"
        "{context}\n\n"
        "Pertanyaan: {question}\n\n"
        "Jawaban:"
    )

    chat_template = ChatPromptTemplate.from_messages([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessagePromptTemplate.from_template(HUMAN_TEMPLATE)
    ])

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | chat_template
        | chat_model
        | StrOutputParser()
    )
    return chain

with st.sidebar:
    st.markdown("<h2 style='font-family:serif; font-size:1.4rem; color:#c9a84c; margin-bottom:0;'>🏠 PROPBOT</h2>", unsafe_allow_html=True)
    st.markdown("<p style='font-size:0.7rem; color:#6b5c3e; letter-spacing:0.15em; text-transform:uppercase; margin-top:0;'>by William Properti</p>", unsafe_allow_html=True)
    st.markdown("<hr style='border-color:rgba(201,168,76,0.2);'/>", unsafe_allow_html=True)
    st.markdown("""
    <div class='contact-card'>
        <p style='margin:0; font-size:0.8rem; font-weight:700;'>📞 HUBUNGI WILLIAM</p>
        <p style='margin:0.3rem 0 0; font-size:1.1rem; font-weight:700;'>123456789</p>
        <p style='margin:0.2rem 0 0; font-size:0.7rem; color:#6b5c3e; text-transform:uppercase; letter-spacing:0.1em;'>Agen Resmi Properti Jawa Barat</p>
    </div>
    """, unsafe_allow_html=True)
    st.markdown("<p style='font-size:0.7rem; letter-spacing:0.1em; text-transform:uppercase; color:#6b5c3e;'>📍 Kota Tersedia</p>", unsafe_allow_html=True)
    for k in ["Bandung","Bekasi","Bogor","Depok","Karawang","Cirebon","Sukabumi","Tasikmalaya","Purwakarta","Cimahi"]:
        st.markdown(f"<p style='font-size:0.82rem; padding:0.1rem 0; margin:0; color:#9b8c6e;'>• {k}</p>", unsafe_allow_html=True)
    st.markdown("<hr style='border-color:rgba(201,168,76,0.2);'/>", unsafe_allow_html=True)
    if st.button("🗑️ Reset Percakapan", use_container_width=True):
        st.session_state.messages = []
        st.rerun()

st.markdown("""
<div class="header-banner">
    <p class="header-title">🏠 William Properti</p>
    <p class="header-subtitle">Katalog Properti Terpercaya Jawa Barat 2026</p>
    <span class="header-badge">✦ PROPBOT Virtual Assistant ✦</span>
</div>
""", unsafe_allow_html=True)

if "messages" not in st.session_state:
    st.session_state.messages = []

with st.spinner("⚙️ Memuat sistem RAG..."):
    try:
        rag_chain = load_rag()
    except Exception as e:
        st.error(f"Gagal memuat RAG: {e}")
        st.stop()

if not st.session_state.messages:
    welcome = "Halo! 👋 Selamat datang di **PROPBOT** — asisten virtual William Properti!\n\nSaya siap membantu Anda menemukan properti impian di **Jawa Barat** 🏡\n\nTanyakan tentang harga, lokasi, cicilan, atau minta rekomendasi sesuai budget Anda!"
    st.session_state.messages.append({"role": "assistant", "content": welcome})

if len(st.session_state.messages) <= 1:
    cols = st.columns(2)
    suggestions = [
        ("🏙️ Properti di Bandung", "Properti apa saja yang tersedia di Bandung?"),
        ("💰 Budget di bawah 1 Miliar", "Rekomendasikan properti dengan harga di bawah 1 miliar rupiah"),
        ("🏗️ Cicilan termurah", "Properti mana yang punya cicilan paling murah?"),
        ("📍 Dekat tol atau stasiun", "Properti apa yang dekat dengan akses tol atau stasiun?"),
    ]
    for i, (label, query) in enumerate(suggestions):
        with cols[i % 2]:
            if st.button(label, use_container_width=True, key=f"sug_{i}"):
                st.session_state.messages.append({"role": "user", "content": query})
                with st.spinner("🔍 Mencari properti..."):
                    response = rag_chain.invoke(query)
                st.session_state.messages.append({"role": "assistant", "content": response})
                st.rerun()

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Tanyakan tentang properti impian Anda..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)
    with st.chat_message("assistant"):
        with st.spinner("🔍 PROPBOT sedang mencari informasi..."):
            try:
                response = rag_chain.invoke(prompt)
            except Exception as e:
                response = f"Maaf, terjadi error: {e}. Silakan hubungi William di 123456789."
        st.markdown(response)
    st.session_state.messages.append({"role": "assistant", "content": response})

Overwriting propbot_app.py


## Cell 6 — Jalankan Streamlit via ngrok

Setelah cell ini jalan, akan muncul URL publik. Buka URL itu di browser untuk akses chatbot.

In [12]:
import subprocess
import time
from pyngrok import ngrok

PORT = 8501

subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
subprocess.run(['fuser', '-k', f'{PORT}/tcp'], capture_output=True)
ngrok.kill()
time.sleep(3)

ngrok.set_auth_token(NGROK_TOKEN)

proc = subprocess.Popen(
    ['streamlit', 'run', 'propbot_app.py',
     '--server.headless=true',
     '--server.port', str(PORT),
     '--server.enableCORS=false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(4)

public_url = ngrok.connect(PORT)
print('=' * 55)
print('🏠 PROPBOT — William Properti Chatbot AKTIF!')
print('=' * 55)
print(f'🌐 Buka URL ini di browser:')
print(f'   {public_url}')
print('=' * 55)
print('💡 Tips pertanyaan:')
print('   - "Properti di Bogor di bawah 1 miliar?"')
print('   - "Bandingkan Meikarta dan Grand Wisata"')
print('   - "Cicilan paling ringan di Depok?"')
print('=' * 55)

🏠 PROPBOT — William Properti Chatbot AKTIF!
🌐 Buka URL ini di browser:
   NgrokTunnel: "https://morse-spirited-gravy.ngrok-free.dev" -> "http://localhost:8501"
💡 Tips pertanyaan:
   - "Properti di Bogor di bawah 1 miliar?"
   - "Bandingkan Meikarta dan Grand Wisata"
   - "Cicilan paling ringan di Depok?"


## Cell 7 — Stop App

Jalankan cell ini kalau mau menghentikan Streamlit dan menutup tunnel ngrok.

In [ ]:
try:
    proc.terminate()
    print('✅ Streamlit dihentikan.')
except:
    print('Tidak ada proses yang berjalan.')

ngrok.kill()
print('✅ Tunnel ngrok ditutup.')